# **Instagram Reels Virality Prediction — Logistic Regression**

## Data Science Lifecycle 기반 프로젝트 (Lecture 2 참고)

본 프로젝트는 강의에서 다룬 **Data Science Lifecycle 6단계**를 따름:
1. **Problem Definition** — Instagram Reels의 바이럴 여부 예측 (Binary Classification)
2. **Data Collection** — Kaggle `What Makes an Instagram Reel Go Viral` 데이터셋
3. **Data Pre-processing** — Cleaning(결측치 처리), Reduction(누수·ID 컬럼 제거), Transformation(Z-score Normalization)
4. **Modeling** — Logistic Regression (Lecture 7의 Single Neuron/Perceptron with Sigmoid)
5. **Evaluation** — Accuracy 외 Cross-entropy 기반 분류 지표
6. **Deployment** — (본 과제 범위 외)

## Problem Definition (Lecture 2)
- **Task Type**: Supervised Learning → Classification → **Binary Classification** (Lecture 6)
- **Target**: `virality_score >= 80` → Viral(1), 아니면 Non-Viral(0)
- **모델 선택 근거 (Lecture 7, 10)**: Logistic Regression은 선형 결합 `z = w·x + b`에 sigmoid를 적용한 단일 퍼셉트론으로, 해석 가능성이 높고 이진 분류의 baseline으로 적합함.

 1. Library Import

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer


2. Random Seed

In [ ]:
RANDOM_STATE = 42


3. Load Dataset


In [ ]:
df = pd.read_csv('/content/instagram data.csv')

 4. Basic Data Check

In [ ]:
print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows")
display(df.head())

print("\nData Info")
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

5. Remove Leakage Features

### 📚 Lecture 5 — Data Reduction (Dimensionality Reduction)
강의의 *Attribute Subset Selection* 관점:
- **Irrelevant attributes**: `reel_id`, `creator_id`는 단순 식별자 → 예측에 무관 (Lecture 5의 "students' ID is often irrelevant" 예시와 동일)
- **Leakage features**: `likes / comments / shares / saves / impressions / reach / engagement_rate`는 `virality_score` 산정에 직·간접적으로 사용되어 target leakage 위험

In [ ]:
drop_columns = [
    'likes',
    'comments',
    'shares',
    'saves',
    'impressions',
    'reach',
    'engagement_rate',
    'reel_id',
    'creator_id'
]

# 존재하는 컬럼만 제거
df = df.drop(
    columns=[col for col in drop_columns if col in df.columns]
)

print("\nRemoved Leakage / ID Columns")


6. Create Target Label
: 제가 모델 돌려봤을 때는 80점은 약한 박한 점수인거 같아서 60점으로 내려서 돌려봤어요 다들 80점으로 최대한 성능 올려보시고 60점으로도 낮춰서 돌려보시면 좋을 것 같습니다!

### 📚 Lecture 6 — Supervised Learning Setup
- 강의의 supervised learning 공식 $y = f(x)$에 맞춰 라벨 $y$ 생성
- **Binary Classification**: `Viral(1)` vs `Non-Viral(0)` (Lecture 6 — Types of Supervised Learning)

In [ ]:
# Viral Score >= 80 -> Viral(1)
# Viral Score < 80 -> Non-Viral(0)
df['viral_label'] = (df['virality_score'] >= 80).astype(int)

# 기존 virality_score 제거
df = df.drop(columns=['virality_score'])

print("\nTarget Label Created")

7. Target Distribution

In [ ]:
print("\nTarget Distribution")
print(df['viral_label'].value_counts())

print("\nTarget Distribution Ratio")
print(df['viral_label'].value_counts(normalize=True))

# 시각화
plt.figure(figsize=(5,4))
sns.countplot(x='viral_label', data=df)
plt.title('Viral Label Distribution')
plt.show()

8. Separate X and y

In [ ]:
X = df.drop(columns=['viral_label'])
y = df['viral_label']

9. Identify Numeric / Categorical Columns

In [ ]:
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

print("\nNumeric Columns")
print(list(numeric_cols))

print("\nCategorical Columns")
print(list(categorical_cols))


 10. Missing Value Handling

### 📚 Lecture 5 — Data Cleaning (Missing Data Handling)
강의의 "Fill in it automatically with the central tendency" 전략 적용:
- 수치형(Numeric): **median** — 이상치에 강건
- 범주형(Categorical): **mode (most_frequent)** — Lecture 5의 central tendency 권장 방식

In [ ]:
# Numeric -> Median Imputation
num_imputer = SimpleImputer(
    strategy='median'
)

X[numeric_cols] = num_imputer.fit_transform(
    X[numeric_cols]
)

# Categorical -> Most Frequent Imputation
if len(categorical_cols) > 0:

    cat_imputer = SimpleImputer(
        strategy='most_frequent'
    )

    X[categorical_cols] = cat_imputer.fit_transform(
        X[categorical_cols]
    )

else:
    print("\nNo categorical columns found.")

print("\nMissing Value Handling Complete")


11. One-Hot Encoding

In [ ]:
if len(categorical_cols) > 0:

    X = pd.get_dummies(
        X,
        columns=categorical_cols,
        drop_first=True
    )

else:
    print("\nNo categorical columns to encode.")

print("\nEncoding Complete")
print("Encoded Data Shape:", X.shape)

12. Correlation Heatmap

13. Numeric Feature Distribution

In [ ]:
for col in numeric_cols[:5]:

    plt.figure(figsize=(6,4))

    sns.histplot(df[col], kde=True)

    plt.title(f'Distribution of {col}')
    plt.show()

14. Boxplot for Outlier Check

In [ ]:
for col in numeric_cols[:5]:

    plt.figure(figsize=(6,4))

    sns.boxplot(x=df[col])

    plt.title(f'Boxplot of {col}')
    plt.show()

15. Train/Test Split

### 📚 Lecture 5 — Sampling (Stratified Sampling)
강의의 *Stratified Sampling* 적용:
- `stratify=y` 옵션으로 train/test의 클래스 비율 유지
- 강의 예시(Machine A 80% / B 20% → 비율대로 표본)와 동일한 논리로 클래스 불균형 환경에서 신뢰 가능한 평가 보장

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("\nTrain/Test Split Complete")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

16. Final Check

In [ ]:
print("\nTrain Label Distribution")
print(y_train.value_counts(normalize=True))

print("\nTest Label Distribution")
print(y_test.value_counts(normalize=True))

print("\nCommon Preprocessing & EDA Complete!")

# **4단계 Modeling — Logistic Regression**

## 📚 Lecture 7, 10 — 이론적 배경

**Logistic Regression = Single Neuron (Perceptron) + Sigmoid Activation (Lecture 7)**

$$z = \mathbf{w}^T \mathbf{x} + b$$

$$h_{w,b}(x) = \sigma(z) = \frac{1}{1 + e^{-z}} = P(y=1 \mid \mathbf{x})$$

- $h_{w,b}(x) \geq \tau$ → class 1 (기본 $\tau=0.5$)
- $h_{w,b}(x) < \tau$ → class 0

**Lecture 10 — Regression Techniques** 중 Logistic Regression은 *linear regression + sigmoid function* 으로 정의됨. 분류 문제에서 출력값(확률)을 0~1로 매핑하여 이진 분류를 수행.

**Loss Function**: Cross-entropy (Lecture 6의 "Choices for measuring Error"에서 언급)

**Regularization (Lecture 10)**: `penalty='l1'`은 **Lasso**, `penalty='l2'`는 **Ridge** 관점에서의 정규화.

## 17. Library Import (Modeling)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, auc, log_loss
)

import warnings
warnings.filterwarnings('ignore')


## 18. Boolean Column Handling

### 📚 Lecture 3 — Quantitative Data
퍼셉트론의 입력 $\mathbf{x}$는 수치형이어야 함. 공통 파이프라인의 `select_dtypes(['int64','float64'])`로는 `bool` 컬럼이 누락되므로 명시적 0/1 변환.

In [ ]:
bool_cols = X_train.select_dtypes(include=['bool']).columns.tolist()
print("Boolean Columns:", bool_cols)

X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

print("\nTrain dtypes (요약):")
print(X_train.dtypes.value_counts())


## 19. Feature Scaling — Z-score Normalization

### 📚 Lecture 5 — Data Transformation (Normalization)
강의의 **Z-score normalization** 공식:

$$v' = \frac{v - \mu_A}{\sigma_A}$$

- 입력 변수 간 스케일 차이가 큼 (`creator_followers` ~ 수백만 vs. `retention_rate` ~ 0–1)
- 퍼셉트론의 가중 합 $z = \mathbf{w}^T \mathbf{x} + b$ 에서 스케일이 큰 변수가 가중치를 과도하게 지배하는 것을 방지
- `fit`은 **train에만** 적용 → test로 누수되지 않도록 함

In [ ]:
scaler = StandardScaler()   # Z-score normalization (Lecture 5)

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Z-score Normalization 적용 완료")
print("(train) mean ≈ 0, std ≈ 1 확인 — 일부 컬럼")
print(X_train_scaled.describe().loc[['mean','std']].T.head())


## 20. Pearson Correlation 분석

### 📚 Lecture 4 — Quantifying Data Relationships
강의의 **Pearson Correlation**으로 입력 변수와 라벨 사이의 선형 관계 강도를 사전 진단.

$$\text{corr}(x, y) = \frac{\text{Cov}(x, y)}{\sigma_x \cdot \sigma_y}$$

In [ ]:
# 라벨과의 Pearson Correlation
corr_with_label = X_train_scaled.copy()
corr_with_label['viral_label'] = y_train.values
pearson_corr = corr_with_label.corr(method='pearson')['viral_label'].drop('viral_label')
pearson_corr = pearson_corr.sort_values(key=lambda s: s.abs(), ascending=False)

print("=== Pearson Correlation with viral_label (Top 10 |corr|) ===")
print(pearson_corr.head(10).round(4))

plt.figure(figsize=(8,5))
pearson_corr.plot(kind='barh', color=['#1f77b4' if v>=0 else '#d62728' for v in pearson_corr])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Pearson Correlation with viral_label')
plt.xlabel('correlation coefficient')
plt.tight_layout()
plt.show()


## 21. Baseline Logistic Regression

기본 설정으로 첫 모델 학습. 강의 Lecture 7의 공식 $h_{w,b}(x) = \sigma(\mathbf{w}^T\mathbf{x} + b)$를 그대로 학습.

In [ ]:
baseline_lr = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    solver='lbfgs'
)
baseline_lr.fit(X_train_scaled, y_train)

y_pred_base  = baseline_lr.predict(X_test_scaled)
y_proba_base = baseline_lr.predict_proba(X_test_scaled)[:, 1]

print("=== Baseline Logistic Regression ===")
print(f"Accuracy        : {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Precision       : {precision_score(y_test, y_pred_base):.4f}")
print(f"Recall          : {recall_score(y_test, y_pred_base):.4f}")
print(f"F1              : {f1_score(y_test, y_pred_base):.4f}")
print(f"ROC-AUC         : {roc_auc_score(y_test, y_proba_base):.4f}")
print(f"Cross-Entropy   : {log_loss(y_test, y_proba_base):.4f}")


## 22. Class Imbalance 대응 — `class_weight='balanced'`

### 📚 Lecture 5 — Stratified Sampling 연계
클래스 비율이 약 80:20 (Non-Viral : Viral)로 불균형. 손실 함수에서 소수 클래스에 가중을 부여하여 강의에서 강조한 "치우친 데이터(skewed)" 대응.

In [ ]:
balanced_lr = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced'
)
balanced_lr.fit(X_train_scaled, y_train)

y_pred_bal  = balanced_lr.predict(X_test_scaled)
y_proba_bal = balanced_lr.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression with class_weight='balanced' ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_bal):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bal):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bal):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bal):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba_bal):.4f}")


## 23. Hyperparameter Tuning — GridSearchCV

### 📚 Lecture 10 — Regression Techniques (Ridge / Lasso)
- `penalty='l1'` ≈ **Lasso** (강의 Lecture 10 회귀 기법 중 하나)
- `penalty='l2'` ≈ **Ridge**
- `C` = 정규화 강도의 역수 (작을수록 강한 정규화)
- 5-fold **Stratified** Cross-Validation으로 검증 (Lecture 5 stratified sampling 원리 적용)

In [ ]:
param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l1', 'l2'],
    'class_weight': [None, 'balanced']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=LogisticRegression(
        solver='liblinear',
        random_state=RANDOM_STATE,
        max_iter=1000
    ),
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_scaled, y_train)

print("\nBest Params :", grid.best_params_)
print(f"Best CV F1  : {grid.best_score_:.4f}")


# **5단계 Evaluation**

### 📚 Lecture 6 — "Choices for measuring Error"
강의에서 언급한 평가지표: **Accuracy, RMSE, ROC, Cross-entropy**.

분류 과제이므로 다음을 사용:
- **Accuracy** — 전체 정확도 (Lecture 6, 7)
- **Precision / Recall / F1** — 클래스 불균형 환경에서 보완 지표
- **ROC-AUC** — Lecture 6에서 언급
- **Cross-Entropy (Log Loss)** — Lecture 6에서 언급, Logistic Regression의 손실함수
- **Confusion Matrix** — 예측의 TP/FP/TN/FN 구조 시각화

## 24. Best Model 종합 평가

In [ ]:
best_lr = grid.best_estimator_

y_pred_best  = best_lr.predict(X_test_scaled)
y_proba_best = best_lr.predict_proba(X_test_scaled)[:, 1]

print("=== Best Tuned Logistic Regression ===")
print(f"Accuracy       : {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Precision      : {precision_score(y_test, y_pred_best):.4f}")
print(f"Recall         : {recall_score(y_test, y_pred_best):.4f}")
print(f"F1             : {f1_score(y_test, y_pred_best):.4f}")
print(f"ROC-AUC        : {roc_auc_score(y_test, y_proba_best):.4f}")
print(f"Cross-Entropy  : {log_loss(y_test, y_proba_best):.4f}")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred_best, target_names=['Non-Viral(0)', 'Viral(1)']))

# 모델별 종합 비교
summary = pd.DataFrame({
    'Baseline': [
        accuracy_score(y_test, y_pred_base),
        precision_score(y_test, y_pred_base),
        recall_score(y_test, y_pred_base),
        f1_score(y_test, y_pred_base),
        roc_auc_score(y_test, y_proba_base),
        log_loss(y_test, y_proba_base),
    ],
    'Balanced': [
        accuracy_score(y_test, y_pred_bal),
        precision_score(y_test, y_pred_bal),
        recall_score(y_test, y_pred_bal),
        f1_score(y_test, y_pred_bal),
        roc_auc_score(y_test, y_proba_bal),
        log_loss(y_test, y_proba_bal),
    ],
    'Tuned(Best)': [
        accuracy_score(y_test, y_pred_best),
        precision_score(y_test, y_pred_best),
        recall_score(y_test, y_pred_best),
        f1_score(y_test, y_pred_best),
        roc_auc_score(y_test, y_proba_best),
        log_loss(y_test, y_proba_best),
    ]
}, index=['Accuracy','Precision','Recall','F1','ROC-AUC','Cross-Entropy'])

print("\n=== Baseline vs Balanced vs Tuned ===")
print(summary.round(4))


## 25. Confusion Matrix
분류 결과의 TP/FP/TN/FN 구조.

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(5,4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Viral','Viral'])
disp.plot(cmap='Blues', ax=ax, colorbar=False)
plt.title('Confusion Matrix — Best Logistic Regression')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Specificity (TN/(TN+FP)) : {tn/(tn+fp):.4f}")
print(f"Sensitivity (Recall)     : {tp/(tp+fn):.4f}")


## 26. ROC Curve
Lecture 6에서 언급된 ROC 지표 시각화.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba_best)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC={roc_auc:.4f})')
plt.plot([0,1], [0,1], linestyle='--', color='gray', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Best Logistic Regression')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 27. 학습된 가중치(Weight) 해석

### 📚 Lecture 7 — Role of Weight & Bias
퍼셉트론의 가중치 $\mathbf{w}$의 부호와 크기로 변수 영향력 해석:
- $w_i > 0$ → 해당 feature가 커질수록 Viral 확률 증가
- $w_i < 0$ → 해당 feature가 커질수록 Viral 확률 감소
- Z-score normalization 후이므로 가중치 절댓값이 곧 상대 영향력

In [ ]:
coef_df = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'weight (w)': best_lr.coef_[0]
})
coef_df['|w|'] = coef_df['weight (w)'].abs()
coef_df = coef_df.sort_values('|w|', ascending=False).reset_index(drop=True)

print("=== Top 15 Features by |weight| ===")
print(coef_df.head(15).to_string(index=False))

top_n = min(15, len(coef_df))
top = coef_df.head(top_n).iloc[::-1]
colors = ['#d62728' if c < 0 else '#1f77b4' for c in top['weight (w)']]

plt.figure(figsize=(8, 6))
plt.barh(top['feature'], top['weight (w)'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('weight (standardized input)')
plt.title(f'Top {top_n} Feature Weights — Logistic Regression (Perceptron)')
plt.tight_layout()
plt.show()

print(f"\nBias (b) = {best_lr.intercept_[0]:.4f}")


## 28. 임계값(Threshold) 비교 실험 — 80점 vs 60점

### 📚 Lecture 7 — Threshold $\tau$ 의 의미
강의의 *"$h_{w,b}(x) \geq \tau$: class 1 (usually $\tau=0.5$ but this is not a strict rule)"* 와 동일한 논리로, 라벨링 임계값(virality_score 기준)도 분석가가 선택하는 의사결정 사항. 80점 / 60점 비교.

In [ ]:
def run_lr_experiment(df_full, threshold, random_state=RANDOM_STATE):
    """공통 전처리 + LR 학습/평가를 임계값별로 재실행."""
    data = df_full.copy()

    # Data Reduction (Lecture 5)
    drop_cols = ['likes','comments','shares','saves','impressions',
                 'reach','engagement_rate','reel_id','creator_id']
    data = data.drop(columns=[c for c in drop_cols if c in data.columns])

    # Label generation (Lecture 6 - supervised setup)
    data['viral_label'] = (data['virality_score'] >= threshold).astype(int)
    data = data.drop(columns=['virality_score'])

    X_ = data.drop(columns=['viral_label'])
    y_ = data['viral_label']

    # Boolean -> int
    bcols = X_.select_dtypes(include=['bool']).columns.tolist()
    X_[bcols] = X_[bcols].astype(int)

    # Stratified Split (Lecture 5)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_, y_, test_size=0.2, stratify=y_, random_state=random_state
    )

    # Z-score Normalization (Lecture 5)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)

    # Logistic Regression (Lecture 7)
    lr = LogisticRegression(
        **{k: v for k, v in grid.best_params_.items()},
        solver='liblinear',
        random_state=random_state,
        max_iter=1000
    )
    lr.fit(X_tr_s, y_tr)

    y_p   = lr.predict(X_te_s)
    y_prb = lr.predict_proba(X_te_s)[:, 1]

    return {
        'threshold': threshold,
        'viral_ratio(train)': round(y_tr.mean(), 4),
        'Accuracy': round(accuracy_score(y_te, y_p), 4),
        'Precision': round(precision_score(y_te, y_p), 4),
        'Recall': round(recall_score(y_te, y_p), 4),
        'F1': round(f1_score(y_te, y_p), 4),
        'ROC-AUC': round(roc_auc_score(y_te, y_prb), 4),
        'CrossEntropy': round(log_loss(y_te, y_prb), 4),
    }

df_raw = pd.read_csv('/content/instagram data.csv')

results = []
for th in [80, 60]:
    r = run_lr_experiment(df_raw, th)
    results.append(r)
    print(f"[Threshold {th}] {r}")

threshold_df = pd.DataFrame(results).set_index('threshold')
print("\n=== Threshold Comparison ===")
print(threshold_df)


# **요약 (Summary)**

## 강의 개념 매핑 (For 발표자료 정리)

| 단계 | 사용 기법 | 근거 강의 |
|---|---|---|
| Problem Definition | Binary Classification 정의 | Lecture 2, 6 |
| Data Reduction | Leakage / ID 컬럼 제거 (Irrelevant Attribute) | Lecture 5 |
| Data Cleaning | Median(numeric) / Mode(categorical) Imputation | Lecture 5 |
| Data Transformation | Z-score Normalization | Lecture 5 |
| Data Relationship | Pearson Correlation 분석 | Lecture 4 |
| Sampling | Stratified Sampling (train/test split) | Lecture 5 |
| Modeling | Logistic Regression = Perceptron + Sigmoid | Lecture 7, 10 |
| Regularization | L1(Lasso) / L2(Ridge) | Lecture 10 |
| Evaluation | Accuracy, ROC, Cross-entropy, Confusion Matrix | Lecture 6 |
| Weight Interpretation | Perceptron의 $\mathbf{w}$, $b$ 역할 | Lecture 7 |

## 결과 정리 포인트 (발표자료 작성용)

1. **공통 파이프라인 (팀 합의 유지)** — Leakage/ID drop, median/mode imputation, stratified split (seed=42, test=0.2), `virality_score ≥ 80` → Viral(1)
2. **개별 LR 처리 (강의 개념 기반)**:
   - Bool → int (퍼셉트론 수치 입력 보장)
   - **Z-score Normalization** (Lecture 5)
   - **Pearson Correlation**으로 변수–라벨 선형 관계 사전 진단 (Lecture 4)
   - **L1/L2 정규화** 및 **class_weight='balanced'** 비교 후 GridSearchCV (5-fold Stratified)로 튜닝
3. **평가 (Lecture 6 지표)**: Accuracy, Precision, Recall, F1, ROC-AUC, **Cross-Entropy** 동시 보고 → 단일 지표 의존 회피
4. **가중치 해석**: 표준화 입력 기준 $|w_i|$ Top 변수로 영향력 정량 분석 (Lecture 7의 Weight 역할)
5. **임계값 실험**: 80점/60점 비교 — Lecture 7의 "$\tau=0.5$는 strict rule이 아님" 원리 확장
